# XRD Rietveld Plot Generator

Turn the CSV your Rietveld refinement exported into a publication figure:
measured pattern, refined fit and background above, reflection positions
underneath, residuals below, saved as a 600 dpi PNG and a vector PDF.

**Check the header of each export once, before you run anything.** GSAS-II
usually writes one header name more than it writes data fields, so every name
sits one column left of its own data. Usually, because the exporter builds
that first column inside a bare `try` and a histogram with no excluded points
makes it raise, so some files arrive aligned. The first cell of the first data
row tells them apart:

| The first cell reads | The file is | Do this |
|---|---|---|
| `1` or `0` | aligned | Nothing |
| your lowest 2θ | shifted | Delete the `used` cell of the header row in a spreadsheet, shift that row one place left |

Do not delete that cell from an aligned file, which would create the shift.
A file still shifted is refused, since the figure would look convincing and be
wrong.

**Then leave the header names alone, except the phase columns.** The pattern
columns are found by their names: `obs` renamed is drawn flat, and a renamed
`diff/sigma` stops the file. The phase columns are yours to rename, and they
are the only place your phase names come from, so `Phase 1` becomes `Rutile`
in the legend by editing the header alone.

**Save the plot on a linear intensity axis.** `obs` is copied from the data,
but `calc`, `bkg` and `diff` are copied from the drawn lines, so a square-root
plot exports the square roots of those three beside an untouched `obs`, with
nothing in the file to mark it.

Do not use the file from *Export → Powder data as → histogram CSV file*. The
other export carries a quoted preamble and different column names, and is
rejected.

**One figure per sample, and one for the series.** Sections 3 and 4 write a
figure per export, which is what most of this notebook is about. Section 5
adds a second kind: the whole series stacked on one set of axes, for the
question a pile of separate figures answers badly, which reflection appears,
moves or goes away across the series. It takes nothing away from the
sections above, and it is driven by two extra columns of
`Samples_metadata.csv` rather than by anything typed into the code.

Full reference: [`README.md`](README.md) and
[`docs/input-format.md`](docs/input-format.md).

## 1. Setup

The dependency check runs first, then the engine. Parsing, data preparation,
plotting and the batch driver live in [`xrd_plotter.py`](xrd_plotter.py),
imported here as `xp`. Appearance comes from the constants at the top of the
file and is overridden on the module, as the cell below shows: the 2θ window,
the colours, the line widths, the font sizes and the residual scale.

Input format and numerical precision are covered in
[`docs/input-format.md`](docs/input-format.md) and
[`docs/validation.md`](docs/validation.md).

In [ ]:
# Dependency bootstrap - installs only what is missing. IPython arrives with
# any Jupyter kernel, and is listed so an editor resolves it as well.
import importlib.util, subprocess, sys

for module, package in (("numpy", "numpy"), ("pandas", "pandas"),
                        ("matplotlib", "matplotlib"),
                        ("ipywidgets", "ipywidgets"), ("IPython", "ipython"),
                        ("pytest", "pytest")):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "--quiet", package])
print("Dependencies OK")

In [ ]:
"""The plotting engine lives in xrd_plotter.py; this cell loads it."""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xrd_plotter as xp

# Appearance is set by the constants in the module. Override them here, on
# the module itself, so every routine sees the change:
#   xp.PLOT_X_MIN, xp.PLOT_X_MAX = 13, 85   # fix the 2theta window
#   xp.WEIGHTED_RESIDUALS = False           # raw obs - calc in the lower panel
#   xp.RESIDUAL_SPAN = 3.0                  # narrower weighted residual panel
#   xp.PHASE_COLORS = {"phase 1": "#1f77b4"}
#   xp.PREVIEW_WIDTH_PX = 500               # smaller inline previews in sec. 3
print("Engine loaded:", Path(xp.__file__).name)

## 2. Validation (self-test on synthetic data)

The cell below runs [`test_xrd_plotter.py`](test_xrd_plotter.py). The suite
builds its own GSAS-II-style exports from an analytic pattern with a fixed
seed, so it reads nothing from `data/` and passes on a fresh clone.

Execution stops at the first failing assertion, so running the notebook is a
test run, and so is `pytest -q` from a terminal. The full list of cases is in
[`docs/validation.md`](docs/validation.md).

In [ ]:
import pytest

# The suite builds its own synthetic exports, so this cell reads nothing
# from data/ and works on a fresh clone. --doctest-modules adds the worked
# examples in the docstrings, which CI runs too. make_series is in the list
# because section 5 draws through it.
exit_code = pytest.main(["-q", "--no-header", "--doctest-modules",
                         "xrd_plotter.py", "test_xrd_plotter.py",
                         "make_series.py", "test_make_series.py"])
assert exit_code == 0, "the validation suite failed, see the report above"
print("\nALL VALIDATION CHECKS PASSED")

## 3. Plot your own exports

Copy your CSV exports into `data/`, put `Samples_metadata.csv` next to the
notebook for real sample names and phase fractions, set the four values
below, and run. Check the header of every export as described at the top of
this notebook first.

Each file prints one block:

- the file name
- the phases found, with the colour each one was drawn in
- the 2θ window drawn, and where it came from: `metadata` from a row of the
  metadata file, `settings` from the `xp.PLOT_X_MIN` and `xp.PLOT_X_MAX`
  constants, `auto` for the full measured range
- the figure
- the two files written, the PDF under `output/pdf/` and the PNG under
  `output/png/`

A file the engine refuses prints `FAILED` with the reason, the run continues,
and every failure is repeated in the summary at the end. A header still one
place out of step gives `FAILED: the 'x, 2theta (deg)' column does not run
from one end of the scan to the other`. Fix the file and run again.

The lower panel is drawn on one scale in every figure. Zero sits at its
middle, and the weighted panel always spans at least ±`xp.RESIDUAL_SPAN`
standard deviations, so two samples stand comparison side by side. A fit
leaving a larger residual widens the panel instead of losing the spike off
the edge.

The `x_min` and `x_max` columns of the metadata file set the window here, per
sample. Section 4 opens each file on the same window and lets you clear a box
to widen back.

> **Keep your data private:** `data/`, `output/` and `Samples_metadata.csv`
> are listed in `.gitignore` and must never be committed or uploaded. The
> procedure is in [`docs/privacy.md`](docs/privacy.md).

In [ ]:
DATA_FOLDER = "data"                       # your GSAS-II CSV exports
METADATA_FILE = "Samples_metadata.csv"     # optional, PRIVATE - never commit
OUTPUT_FOLDER = "output"                   # created on the first run
USE_SQRT = True                            # False -> linear intensity axis

results = xp.process_folder(DATA_FOLDER, METADATA_FILE, OUTPUT_FOLDER,
                         use_sqrt=USE_SQRT)

## 4. Try a different window on one file

Pick a file and type limits. The figure updates live and in place: a text box
on Enter or when you leave it, a checkbox or the dropdown at once. Picking a
file fills the 2θ boxes from its metadata row, so it opens on the window
section 3 draws. Clear a 2θ box and that end of the axis falls back to the
`xp.PLOT_X_MIN` and `xp.PLOT_X_MAX` constants, which is the full measured
range while they are `None`. An empty intensity box leaves that end to the
data.

**Save to output** writes the window on screen to `output/` at full
resolution, under the name the batch uses, so you settle a window here without
rerunning section 3. The two checkboxes are part of that name: untick one and
the figure is written as `_linear` or `_unweighted`, so a preview never
overwrites a batch figure. Every other control previews only.

The line above the figure carries the engine messages for the file and the
metadata row for the window on screen. Paste the row into
`Samples_metadata.csv` and section 3 draws the sample this way on every run.

This section needs `ipywidgets`, installed by the first cell. Without it the
section prints how to install it, and the rest of the notebook is
unaffected.

In [ ]:
# No widgets.Output here. An Output cleared inside a callback appends a second
# figure under the first in VS Code, so the panel would fill with stale plots.
# The figure is a widgets.Image and the log a widgets.HTML, both value
# replaced, so every change updates the same two areas in place, never appends.
import contextlib
import html
import io
import traceback

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    print("ipywidgets is not installed: run 'pip install ipywidgets', "
          "then re-run this cell.")

files = sorted(f for f in Path(DATA_FOLDER).glob("*.csv")
               if f.name != Path(METADATA_FILE).name)


def pre(text):
    """Escaped monospace block for a widgets.HTML value."""
    return ("<pre style='margin:0;font:12px/1.4 monospace;white-space:pre-wrap'>"
            f"{html.escape(text)}</pre>")


if widgets is None or not files:
    if widgets is not None:
        print(f"No CSV files in '{DATA_FOLDER}': nothing to replot.")
else:
    picker = widgets.Dropdown(options=[(f.name, str(f)) for f in files],
                              description="File:",
                              layout=widgets.Layout(width="420px"))
    # continuous_update=False: a box redraws when you press Enter or leave
    # it, not on every keystroke.
    boxes = {k: widgets.Text(description=d, placeholder="auto",
                             continuous_update=False,
                             layout=widgets.Layout(width="180px"))
             for k, d in (("x_min", "2theta min"), ("x_max", "2theta max"),
                          ("y_min", "y min"), ("y_max", "y max"))}
    sqrt_box = widgets.Checkbox(value=USE_SQRT, description="sqrt intensity")
    weighted_box = widgets.Checkbox(value=xp.WEIGHTED_RESIDUALS,
                                    description="diff/sigma")
    save_button = widgets.Button(description="Save to output",
                                 button_style="success", icon="download")
    status = widgets.HTML()
    canvas = widgets.Image(format="png",
                           layout=widgets.Layout(width="100%",
                                                 max_width="820px"))
    busy = [False]
    filling = [False]  # True while a file pick rewrites the 2theta boxes

    def draw_current():
        """Render the picked file with whatever the controls now hold.

        Returns (fig, metadata_line, log_text). Raises ValueError when the
        file cannot be drawn. The caller decides whether to preview or save.
        """
        limits = {k: xp.to_number(b.value) if b.value.strip() else None
                  for k, b in boxes.items()}
        log = io.StringIO()
        # Engine messages belong in the status block. Left on stdout they
        # land under the cell and pile up one copy per redraw.
        with contextlib.redirect_stdout(log):
            fig, line = xp.replot_file(picker.value, METADATA_FILE,
                                       use_sqrt=sqrt_box.value,
                                       weighted=weighted_box.value, **limits)
        return fig, line, log.getvalue()

    def redraw(_=None):
        """Update the on-screen preview in place; saves nothing."""
        if filling[0] or busy[0]:
            return  # skip the box writes of a file pick, and re-entrancy
        busy[0] = True
        try:
            try:
                fig, line, log = draw_current()
            except ValueError as e:
                # Hide the stale figure: it belongs to a different file, and
                # leaving it up invites new limits typed against it.
                canvas.layout.display = "none"
                status.value = pre(f"Cannot draw this file: {e}")
                return
            png = io.BytesIO()
            try:
                # 110 dpi is a screen preview. Save writes the 600 dpi file.
                fig.savefig(png, format="png", dpi=110, bbox_inches="tight",
                            facecolor="white")
            finally:
                plt.close(fig)  # a failed render must not leak the figure
            canvas.value = png.getvalue()
            canvas.layout.display = ""
            status.value = pre(f"{log}Metadata line for this window:\n{line}")
        except Exception:
            plt.close("all")
            canvas.layout.display = "none"
            status.value = pre("Redraw failed:\n" + traceback.format_exc())
        finally:
            busy[0] = False

    def on_pick(_=None):
        """Prefill the 2theta boxes from the file's metadata, then redraw.

        The boxes open on the window the batch would use, so a peak cropped
        out in section 3 is cropped here too. Clear a box to widen back.
        """
        filling[0] = True
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                x_min, x_max = xp.sample_window(METADATA_FILE,
                                                Path(picker.value).name)
            boxes["x_min"].value = "" if x_min is None else f"{x_min:g}"
            boxes["x_max"].value = "" if x_max is None else f"{x_max:g}"
        finally:
            filling[0] = False
        redraw()

    def save(_=None):
        """Write the current window to output/pdf and output/png."""
        if busy[0]:
            return
        busy[0] = True
        try:
            try:
                fig, _line, log = draw_current()
            except ValueError as e:
                status.value = pre(f"Cannot draw this file: {e}")
                return
            try:
                base = xp.output_basename(Path(picker.value).stem,
                                          sqrt_box.value, weighted_box.value)
                xp.save_figure(fig, OUTPUT_FOLDER, base)
            finally:
                plt.close(fig)
            status.value = pre(f"{log}Saved pdf/{base}.pdf and png/{base}.png")
        except Exception:
            plt.close("all")
            status.value = pre("Save failed:\n" + traceback.format_exc())
        finally:
            busy[0] = False

    # Picking a file refills the 2theta boxes and redraws; every other
    # control redraws the one figure in place. Only the button writes to disk.
    picker.observe(on_pick, names="value")
    for control in (sqrt_box, weighted_box, *boxes.values()):
        control.observe(redraw, names="value")
    save_button.on_click(save)

    display(widgets.VBox([
        picker,
        widgets.HBox([boxes["x_min"], boxes["x_max"]]),
        widgets.HBox([boxes["y_min"], boxes["y_max"]]),
        widgets.HBox([sqrt_box, weighted_box, save_button]),
        status,
        canvas,
    ]))
    on_pick()  # open on the first file, boxes prefilled from its metadata


## 5. The whole series as one figure

Every sample that carries a number in the `series_order` column of
`Samples_metadata.csv` is drawn here as one trace, in that order, bottom to
top, with `series_label` naming it. Nothing about your samples lives in the
code: reorder the series in the spreadsheet and run this cell again. A sample
with no number stays out of the series and keeps the figure section 3 writes
for it, which this section neither replaces nor overwrites.

Each pattern is rescaled to the same height, so a weak sample stays visible
beside a strong one and **no height in this figure may be compared with
another**. The controls below redraw the preview in place. *Save to output*
writes the PDF and the PNG under `output/`, and nothing is written until you
press it.

The 2θ window applies to every trace, which is what keeps them comparable.
The tick rows and the guides are read from the first member of the series, so
in a series drawn precisely because a reflection moves, they mark where it
started rather than where each trace has it. Type the guides as 2θ separated
by commas: each one snaps to the nearest reflection and is drawn in the
colour of the phase that owns it, and the positions to pick from are printed
under the figure, so you choose from the reflections that are there rather
than reading them off the picture by eye.

> **`GUIDE_LINES` in `make_series.py` is the one setting there that carries a
> measurement.** A reflection position gives a lattice spacing, so leave the
> guides in the box here, where they never reach a tracked file. See
> [`docs/privacy.md`](docs/privacy.md).

This section needs `ipywidgets`, installed by the first cell. Without it the
section prints how to install it, and the rest of the notebook is
unaffected.

In [ ]:
# Section 4's rules apply here for the same reasons: no widgets.Output, the
# figure is a value-replaced widgets.Image and the log a widgets.HTML, so a
# redraw never appends a second copy under the first. Every name here starts
# with 'series_', because section 4's callbacks read their controls off the
# globals and a shared name would make that section read this one's widgets.
import contextlib
import html
import io
import traceback

import make_series as ms

# Imported again rather than borrowed from section 4, so this cell runs on
# its own after section 1. Both names bind to the same modules.
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    print("ipywidgets is not installed: run 'pip install ipywidgets', "
          "then re-run this cell.")

series_meta = xp.load_metadata(METADATA_FILE)
series_entries = ms.series_from_metadata(series_meta)


def series_pre(text):
    """Escaped monospace block for a widgets.HTML value."""
    return ("<pre style='margin:0;font:12px/1.4 monospace;"
            f"white-space:pre-wrap'>{html.escape(text)}</pre>")


if widgets is None or not series_entries:
    if widgets is not None:
        print(f"No sample carries a 'series_order' in {METADATA_FILE}: "
              "nothing to stack. See docs/metadata.md.")
else:
    series_numbers = {
        k: widgets.Text(description=d, placeholder="auto",
                        continuous_update=False,
                        layout=widgets.Layout(width="200px"))
        for k, d in (("x_min", "2theta min"), ("x_max", "2theta max"),
                     ("offset", "offset"), ("label_height", "label y"),
                     ("label_x", "label 2theta"),
                     ("tick_height", "tick height"))}
    series_guides_box = widgets.Text(description="guides",
                                     placeholder="31.0, 37.04",
                                     continuous_update=False,
                                     layout=widgets.Layout(width="420px"))
    series_weight_box = widgets.Dropdown(
        options=["normal", "medium", "semibold", "bold"],
        value=ms.LABEL_WEIGHT, description="weight",
        layout=widgets.Layout(width="200px"))
    series_sqrt_box = widgets.Checkbox(value=ms.USE_SQRT,
                                       description="sqrt intensity")
    series_ticks_box = widgets.Checkbox(value=ms.SHOW_TICKS,
                                        description="ticks")
    series_save_button = widgets.Button(description="Save to output",
                                        button_style="success",
                                        icon="download")
    series_status = widgets.HTML()
    series_canvas = widgets.Image(format="png",
                                  layout=widgets.Layout(width="100%",
                                                        max_width="820px"))
    series_busy = [False]

    def series_number(key):
        """One box as a float, or None when it is blank."""
        text = series_numbers[key].value.strip()
        return xp.to_number(text) if text else None

    def draw_series():
        """Draw the series with whatever the controls now hold.

        Returns (fig, log_text). The caller decides whether to preview it or
        save it, so nothing here writes a file.
        """
        guides = [xp.to_number(part)
                  for part in series_guides_box.value.split(",")
                  if part.strip()]
        label_x = series_number("label_x")
        log = io.StringIO()
        # Engine and script messages belong in the status block. Left on
        # stdout they land under the cell, one copy per redraw.
        with contextlib.redirect_stdout(log):
            # The same use_sqrt reaches both calls: load_series applies the
            # transform, plot_series only names the axis, and passing them
            # apart would label a root nobody took.
            traces, phases, colors = ms.load_series(
                series_entries, DATA_FOLDER, series_meta,
                use_sqrt=series_sqrt_box.value)
            fig = ms.plot_series(
                traces, phases, colors=colors,
                offset=series_number("offset"),
                label_height=series_number("label_height"),
                # A blank box means the left border, which plot_series
                # spells as NaN, since None already means 'take the setting'.
                label_x=float("nan") if label_x is None else label_x,
                label_weight=series_weight_box.value,
                tick_height=series_number("tick_height"),
                show_ticks=series_ticks_box.value,
                guide_lines=[g for g in guides if g is not None],
                use_sqrt=series_sqrt_box.value,
                window=(series_number("x_min"), series_number("x_max")))
        return fig, log.getvalue()

    def series_redraw(_=None):
        """Update the preview in place; saves nothing."""
        if series_busy[0]:
            return
        series_busy[0] = True
        try:
            try:
                fig, log = draw_series()
            except (ValueError, SystemExit) as e:
                # SystemExit as well as ValueError: load_series raises it
                # when a member of the series is missing from the folder,
                # and it is not an Exception, so nothing below would catch
                # it and the cell would stop instead of reporting.
                series_canvas.layout.display = "none"
                series_status.value = series_pre(f"Cannot draw: {e}")
                return
            png = io.BytesIO()
            try:
                fig.savefig(png, format="png", dpi=110, bbox_inches="tight",
                            facecolor="white")
            finally:
                plt.close(fig)
            series_canvas.value = png.getvalue()
            series_canvas.layout.display = ""
            series_status.value = series_pre(log or "Drawn.")
        except Exception:
            plt.close("all")
            series_canvas.layout.display = "none"
            series_status.value = series_pre("Redraw failed:\n"
                                             + traceback.format_exc())
        finally:
            series_busy[0] = False

    def series_save(_=None):
        """Write the series as it now stands to output/pdf and output/png."""
        if series_busy[0]:
            return
        series_busy[0] = True
        try:
            try:
                fig, log = draw_series()
            except (ValueError, SystemExit) as e:
                series_status.value = series_pre(f"Cannot draw: {e}")
                return
            try:
                # weighted=True: the series draws no residual, so the
                # unweighted suffix would mean nothing here. The sqrt
                # setting still splits the two names apart.
                base = xp.output_basename(ms.OUTPUT_BASENAME,
                                          series_sqrt_box.value,
                                          weighted=True)
                xp.save_figure(fig, OUTPUT_FOLDER, base)
            finally:
                plt.close(fig)
            series_status.value = series_pre(
                f"{log}Saved pdf/{base}.pdf and png/{base}.png")
        except Exception:
            plt.close("all")
            series_status.value = series_pre("Save failed:\n"
                                             + traceback.format_exc())
        finally:
            series_busy[0] = False

    for series_control in (*series_numbers.values(), series_guides_box,
                           series_weight_box, series_sqrt_box,
                           series_ticks_box):
        series_control.observe(series_redraw, names="value")
    series_save_button.on_click(series_save)

    display(widgets.VBox([
        widgets.HBox([series_numbers["x_min"], series_numbers["x_max"],
                      series_sqrt_box, series_ticks_box]),
        widgets.HBox([series_numbers["offset"],
                      series_numbers["label_height"],
                      series_numbers["label_x"], series_weight_box]),
        widgets.HBox([series_numbers["tick_height"], series_guides_box,
                      series_save_button]),
        series_status,
        series_canvas,
    ]))
    series_redraw()